# ROGII Wellbore Geology Prediction
Clean Kaggle notebook with data-driven ensemble weights.

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd

from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True

print("Imports loaded")

## Data discovery and helpers

In [ ]:
def find_input_dir():
    candidates = [
        "/kaggle/input/rogii-wellbore-geology-prediction",
        "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    ]

    for c in candidates:
        if os.path.isdir(c):
            print(f"INPUT_DIR = {c}")
            return c

    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    if hits:
        d = os.path.dirname(hits[0])
        print(f"Discovered INPUT_DIR = {d}")
        return d

    raise FileNotFoundError("Cannot locate competition data")


INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, "train")
TEST_DIR = os.path.join(INPUT_DIR, "test")

_hw_tr = sorted(glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv")))
_hw_te = sorted(glob.glob(os.path.join(TEST_DIR, "*__horizontal_well.csv")))

TRAIN_WELLS = [os.path.basename(f).split("__")[0] for f in _hw_tr]
TEST_WELLS = [os.path.basename(f).split("__")[0] for f in _hw_te]

print(f"Train wells: {len(TRAIN_WELLS)}")
print(f"Test wells : {len(TEST_WELLS)}")
print("Test well IDs:", TEST_WELLS[:10])


def load_well(wid, split="train"):
    base = TRAIN_DIR if split == "train" else TEST_DIR

    hw_path = os.path.join(base, f"{wid}__horizontal_well.csv")
    hw = pd.read_csv(hw_path)

    tw_files = glob.glob(os.path.join(base, f"{wid}__typewell*.csv"))
    if len(tw_files) == 0:
        raise FileNotFoundError(f"No typewell file found for {wid} in {split}")

    tw = pd.read_csv(tw_files[0])
    return hw, tw


def well_centroid(wid, split="train"):
    hw, _ = load_well(wid, split)
    return np.array([hw["X"].mean(), hw["Y"].mean()], dtype=float)

## Physical model from geological contacts

In [ ]:
def tvt_from_contacts(hw_tr, tw_tr, ref_col=None, verbose=False):
    """
    Compute TVT from geological contact depths for a visible training well.
    Automatically selects the best formation contact if ref_col is None.
    """
    tw_g = tw_tr.dropna(subset=["Geology"]).copy()
    kn = hw_tr[hw_tr["TVT_input"].notna()].copy()

    if len(kn) == 0:
        raise ValueError("No known TVT_input rows found.")

    candidates = [col for col in tw_g["Geology"].dropna().unique() if col in hw_tr.columns]
    if len(candidates) == 0:
        raise ValueError("No valid geology contact columns found in the horizontal well.")

    def _fit_col(col):
        ref_rows = tw_g[tw_g["Geology"] == col]
        if len(ref_rows) == 0:
            return None, np.inf

        ref_tvt = ref_rows["TVT"].min()
        if not np.isfinite(ref_tvt):
            return None, np.inf

        pred = ref_tvt - (hw_tr["Z"] - hw_tr[col])
        offset = (kn["TVT_input"] - pred.loc[kn.index]).mean()
        pred = pred + offset

        rmse = np.sqrt(((kn["TVT_input"] - pred.loc[kn.index]) ** 2).mean())
        return pred, rmse

    if ref_col is not None:
        pred, _ = _fit_col(ref_col)
        if pred is not None:
            if verbose:
                print(f"Using ref_col={ref_col}")
            return pred

    best_pred = None
    best_rmse = np.inf
    best_col = None

    for col in candidates:
        pred, rmse = _fit_col(col)
        if pred is not None and rmse < best_rmse:
            best_pred = pred
            best_rmse = rmse
            best_col = col

    if best_pred is not None:
        if verbose:
            print(f"Selected formation: {best_col}, RMSE={best_rmse:.4f}")
        return best_pred

    raise ValueError("No valid geology formation column found.")

## GR template matching (k-NN)

In [ ]:
def gr_knn_tvt(hw, k=10, window=11, sigma_scale=1.0, blend=0.7):
    """
    k-NN TVT prediction using pre-PS GR as a local correlation template.
    """
    kn = hw[hw["TVT_input"].notna()].copy()
    ev = hw[hw["TVT_input"].isna()].copy()

    if len(ev) == 0:
        return hw["TVT_input"].values.astype(float).copy()

    gr_full = (
        hw["GR"]
        .interpolate(limit_direction="both")
        .fillna(hw["GR"].median())
        .values.astype(float)
    )

    n = len(gr_full)
    kn_idx = list(kn.index)
    ev_idx = list(ev.index)
    half = window

    def get_win(arr, i, half_window):
        start_idx = i - half_window
        end_idx = i + half_window + 1

        pad_left = max(0, -start_idx)
        pad_right = max(0, end_idx - n)

        slice_start = max(0, start_idx)
        slice_end = min(n, end_idx)

        window_slice = arr[slice_start:slice_end]
        return np.pad(window_slice, (pad_left, pad_right), mode="edge")

    kn_wins = np.stack([get_win(gr_full, i, half) for i in kn_idx])
    ev_wins = np.stack([get_win(gr_full, i, half) for i in ev_idx])

    def row_norm(X):
        mu = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True) + 1e-6
        return (X - mu) / std

    kn_wins_n = row_norm(kn_wins)
    ev_wins_n = row_norm(ev_wins)

    kn_tvt = kn["TVT_input"].values.astype(float)

    D = cdist(ev_wins_n, kn_wins_n, metric="euclidean")
    last_tvt = float(kn_tvt[-1])
    last_md = float(kn["MD"].iloc[-1])

    if len(kn) >= 5:
        recent = kn.tail(10)
        md_diff = np.diff(recent["MD"].values.astype(float))
        tvt_diff = np.diff(recent["TVT_input"].values.astype(float))
        valid = md_diff > 0
        local_slope = float(np.median(tvt_diff[valid] / md_diff[valid])) if valid.any() else 0.0
    else:
        local_slope = 0.0

    out = hw["TVT_input"].values.astype(float).copy()

    for i, idx in enumerate(ev_idx):
        nn = np.argsort(D[i])[:k]
        d = D[i][nn]

        sigma = max(sigma_scale * np.median(d), 1e-3)
        wts = np.exp(-(d ** 2) / (2 * sigma ** 2))
        wts /= wts.sum()

        knn_pred = np.dot(wts, kn_tvt[nn])

        md_offset = float(hw.loc[idx, "MD"] - last_md)
        trend_pred = last_tvt + local_slope * md_offset

        out[idx] = blend * knn_pred + (1.0 - blend) * trend_pred

    return out

## Particle filter ensemble

In [ ]:
def run_particle_filter(hw, tw, n_particles=500, seed=42):
    # Particle filter. Returns (predictions_array, total_log_likelihood).

    tw_s = tw.sort_values("TVT").copy()
    tw_tvt = tw_s["TVT"].values.astype(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).values.astype(float)

    kn = hw[hw["TVT_input"].notna()].copy()
    ev = hw[hw["TVT_input"].isna()].copy()

    if len(ev) == 0:
        return hw["TVT_input"].values.astype(float).copy(), 0.0

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"])
    last_Z = float(last["Z"])
    last_MD = float(last["MD"])

    tw_at_k = np.interp(kn["TVT_input"].values, tw_tvt, tw_gr)
    gr_resid = kn["GR"].fillna(kn["GR"].median()).values.astype(float) - tw_at_k
    gs = float(np.clip(np.nanstd(gr_resid), 5.0, 80.0))
    if not np.isfinite(gs) or gs <= 0:
        gs = 20.0

    tail = kn.tail(30)
    if len(tail) >= 3:
        dt = np.diff(tail["TVT_input"].values.astype(float))
        dz = np.diff(tail["Z"].values.astype(float))
        dm = np.diff(tail["MD"].values.astype(float))
        valid = dm > 0
        ir = float(np.median((dt + dz)[valid] / dm[valid])) if valid.any() else 0.0
    else:
        ir = 0.0

    N = n_particles
    rng = np.random.default_rng(seed)

    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N, dtype=float) / N

    MOM = 0.998
    VN = 0.002
    PN = 0.005
    RP = 0.1
    RR = 0.001
    RESAMP = 0.5

    md_v = ev["MD"].values.astype(float)
    z_v = ev["Z"].values.astype(float)
    gr_v = ev["GR"].values.astype(float)

    out_vals = hw["TVT_input"].values.astype(float).copy()
    res = np.empty(len(ev), dtype=float)
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)

        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm_step + PN * rng.standard_normal(N)

        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos = tvt_p + z_v[i]

        if not np.isnan(gr_v[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d ** 2, 600.0))
            lk = np.maximum(lk, 1e-300)

            avg_lk = float((w * lk).sum())
            log_lik += np.log(max(avg_lk, 1e-300))

            w = w * lk
            ws = w.sum()
            if ws > 0:
                w /= ws
            else:
                w = np.ones(N, dtype=float) / N

        n_eff = 1.0 / np.sum(w ** 2)
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N, dtype=float) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    # Likelihood-weighted PF ensemble.
    preds = []
    liks = []

    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks = np.array(liks, dtype=float)
    liks_n = liks - np.max(liks)

    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, axis=0)).sum(axis=0)

## Beam search ensemble

In [ ]:
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5), (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2), (15, 10.0, 80.0, 4),
    (25, 6.0, 50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
    (30, 8.0, 70.0, 2), (10, 50.0, 400.0, 0),
]


def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n = len(hgr)
    nt = len(tw_tvt)

    if n == 0:
        return np.array([last_tvt], dtype=float)

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC = mc * np.array([2.0, 1.0, 0.0, 1.0, 2.0])

    bidx = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf, dtype=float)
    bcost[0] = 0.0
    bn = 1

    result = np.zeros(n, dtype=float)

    for step in range(n):
        gv = sgr[step]

        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci]) ** 2 / es
        tot = bcost[:bn, None] + gr_e + MC[None, :]
        tot = np.where(valid, tot, np.inf)

        valid_flat = valid.flatten()
        ni_f = ni.flatten()[valid_flat]
        tot_f = tot.flatten()[valid_flat]

        if len(tot_f) == 0:
            result[step] = tw_tvt[bidx[0]]
            continue

        order = np.argsort(tot_f)
        ni_s = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top = np.argsort(tot_u)[:kept]

        bidx[:kept] = ni_u[top]
        bcost[:kept] = tot_u[top]

        if kept < bs:
            bidx[kept:] = bidx[kept - 1]
            bcost[kept:] = np.inf

        bn = kept
        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    # Average 14 beam-search configs.
    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]

    if len(ev) == 0:
        return hw["TVT_input"].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]["TVT_input"])

    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].values.astype(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).values.astype(float)

    gr_all = (
        hw["GR"]
        .interpolate(limit_direction="both")
        .fillna(tw_gr.mean())
        .values.astype(float)
    )

    hgr = gr_all[ev.index.to_numpy()]

    beam_results = [
        beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
        for (bs, mc, es, r) in BEAM_CONFIGS
    ]

    beam_mean = np.stack(beam_results, axis=0).mean(axis=0)

    out = hw["TVT_input"].values.astype(float).copy()
    out[ev.index.to_numpy()] = beam_mean
    return out

## Offset-well prior

In [ ]:
# Pre-compute training well centroids once
_train_centroids = {}
for _wid in TRAIN_WELLS:
    try:
        _train_centroids[_wid] = well_centroid(_wid, "train")
    except Exception:
        pass


def find_offset_wells(test_wid, split="test", n_offset=5):
    """Return the n nearest training well IDs sorted by XY distance."""
    try:
        c_test = np.array(well_centroid(test_wid, split), dtype=float)
    except Exception:
        return []

    dists = []
    for wid, c in _train_centroids.items():
        if split == "train" and wid == test_wid:
            continue
        d = float(np.linalg.norm(c_test - np.array(c, dtype=float)))
        dists.append((wid, d))

    dists.sort(key=lambda x: x[1])
    return [wid for wid, _ in dists[:n_offset]]


def offset_well_prior(hw_te, offset_wells, n_offset=5):
    """
    Linear TVT extrapolation anchored to the last known TVT,
    with dip rate (dTVT/dMD) estimated from neighboring training wells.
    """
    kn = hw_te[hw_te["TVT_input"].notna()].copy()
    ev = hw_te[hw_te["TVT_input"].isna()].copy()

    if len(ev) == 0 or len(kn) == 0 or not offset_wells:
        return None

    rates = []

    for wid in offset_wells:
        try:
            hw_off, _ = load_well(wid, "train")
            kn_off = hw_off[hw_off["TVT_input"].notna()].copy()

            if len(kn_off) < 10:
                continue

            tail = kn_off.tail(60)
            dtvt = np.diff(tail["TVT_input"].values.astype(float))
            dmd = np.diff(tail["MD"].values.astype(float))
            valid = dmd > 0

            if valid.sum() >= 5:
                rates.append(float(np.median(dtvt[valid] / dmd[valid])))

        except Exception:
            pass

    if not rates:
        return None

    med_rate = float(np.median(rates))

    last_tvt = float(kn.iloc[-1]["TVT_input"])
    last_md = float(kn.iloc[-1]["MD"])

    out = hw_te["TVT_input"].values.astype(float).copy()

    for idx in ev.index:
        md_delta = float(hw_te.loc[idx, "MD"] - last_md)
        out[idx] = last_tvt + med_rate * md_delta

    return out

## Cross-validation calibration and data-driven weights

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


METHOD_NAMES = ["pf", "beam", "knn", "offset"]

CV_WELL_LIMIT = min(12, len(TRAIN_WELLS))
rng = np.random.default_rng(SEED)
CV_WELLS = list(rng.choice(TRAIN_WELLS, size=CV_WELL_LIMIT, replace=False))
CV_WELLS.sort()

print("Calibration wells:", CV_WELLS)

cv_results = {}

# Lightweight settings for calibration
CV_PF_PARTICLES = 200
CV_PF_SEEDS = 24

for wid in CV_WELLS:
    print(f"CV well: {wid}")
    hw, tw = load_well(wid, "train")
    ev = hw[hw["TVT_input"].isna()].copy()

    if len(ev) == 0:
        print("  no prediction zone, skipped")
        continue

    cv_results[wid] = {}

    preds = {}
    preds["pf"] = run_pf_ensemble(hw, tw, n_particles=CV_PF_PARTICLES, n_seeds=CV_PF_SEEDS, scale=5.0)
    preds["beam"] = run_beam_ensemble(hw, tw)
    preds["knn"] = gr_knn_tvt(hw, k=10, window=11, sigma_scale=1.0, blend=0.7)

    offs = find_offset_wells(wid, split="train", n_offset=5)
    preds["offset"] = offset_well_prior(hw, offs)

    y_true = hw.loc[ev.index, "TVT"].values.astype(float)

    for m, p in preds.items():
        if p is None:
            continue
        cv_results[wid][m] = rmse(y_true, p[ev.index])

    print(" ", cv_results[wid])


mean_rmse = {}
for method in METHOD_NAMES:
    vals = [
        cv_results[w][method]
        for w in cv_results
        if method in cv_results[w]
        and cv_results[w][method] is not None
        and np.isfinite(cv_results[w][method])
    ]
    if len(vals) > 0:
        mean_rmse[method] = float(np.mean(vals))

print("=" * 50)
print("Mean RMSE per method")
print("=" * 50)
for method, score in sorted(mean_rmse.items(), key=lambda x: x[1]):
    print(f"{method:10s} : {score:.4f} ft")

T = 0.5
available = {m: v for m, v in mean_rmse.items() if np.isfinite(v)}

if available:
    methods_s = sorted(available)
    rmse_arr = np.array([available[m] for m in methods_s], dtype=float)
    rmse_arr = rmse_arr - rmse_arr.min()
    weights = np.exp(-rmse_arr / T)
    weights /= weights.sum()
    ENSEMBLE_WEIGHTS = dict(zip(methods_s, weights))
else:
    ENSEMBLE_WEIGHTS = {m: 1.0 / len(METHOD_NAMES) for m in METHOD_NAMES}

print("=" * 50)
print("CV-Tuned Ensemble Weights")
print("=" * 50)
for m, w in sorted(ENSEMBLE_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"{m:10s}: {w:.4f}")

if len(mean_rmse) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    methods = list(mean_rmse.keys())
    scores = [mean_rmse[m] for m in methods]
    best = min(scores)
    colors = ["steelblue" if s == best else "salmon" for s in scores]
    bars = ax.bar(methods, scores, color=colors)
    ax.set_ylabel("Mean RMSE (ft)")
    ax.set_title("Cross-Validated RMSE by Method")
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, score, f"{score:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()

## Final submission generation

In [ ]:
sample = pd.read_csv(os.path.join(INPUT_DIR, "sample_submission.csv"))
sample["well"] = sample["id"].str[:8]
sample["row_idx"] = sample["id"].str.split("_").str[-1].astype(int)

train_wids = set(TRAIN_WELLS)
print(f"Training wells available: {len(train_wids)}")


def blend_hidden(preds_dict):
    """Weighted average of method predictions using CV-tuned weights."""
    total_w = 0.0
    total_p = None

    for m, arr in preds_dict.items():
        if arr is None:
            continue

        w = float(ENSEMBLE_WEIGHTS.get(m, 0.0))
        if w <= 0:
            continue

        arr = np.asarray(arr, dtype=float)
        total_p = arr * w if total_p is None else total_p + arr * w
        total_w += w

    if total_p is None:
        raise ValueError("All methods returned None")

    return total_p / total_w if total_w > 0 else total_p


rows = []

for wid in TEST_WELLS:
    print(f"\nProcessing {wid}...")
    hw_te, tw_te = load_well(wid, "test")

    tvt_phys = None
    hw_tr = None
    tw_tr = None

    if wid in train_wids:
        try:
            hw_tr, tw_tr = load_well(wid, "train")
            hw_te["TVT_input"] = hw_tr["TVT_input"].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print("  Physical model OK (auto formation)")
        except Exception as e:
            print(f"  Physical model failed: {e}")

    tw_ref = tw_tr if tw_tr is not None else tw_te
    preds = {}

    try:
        preds["pf"] = run_pf_ensemble(
            hw_te, tw_ref,
            n_particles=500,
            n_seeds=128,
            scale=5.0
        )
        print("  PF ensemble OK")
    except Exception as e:
        print(f"  PF failed: {e}")
        lk = hw_te["TVT_input"].dropna()
        last_val = float(lk.iloc[-1]) if len(lk) else 0.0
        preds["pf"] = hw_te["TVT_input"].fillna(last_val).values.astype(float)

    try:
        preds["beam"] = run_beam_ensemble(hw_te, tw_ref)
        print("  Beam ensemble OK")
    except Exception as e:
        print(f"  Beam failed: {e}")
        preds["beam"] = preds["pf"].copy()

    try:
        preds["knn"] = gr_knn_tvt(hw_te, k=10, window=11, sigma_scale=1.0, blend=0.7)
        print("  GR k-NN OK")
    except Exception as e:
        print(f"  GR k-NN failed: {e}")
        preds["knn"] = None

    try:
        offs = find_offset_wells(wid, split="test", n_offset=5)
        prior = offset_well_prior(hw_te, offs)
        if prior is not None:
            preds["offset"] = prior
            print(f"  Offset prior OK (neighbors: {offs})")
        else:
            preds["offset"] = None
    except Exception as e:
        print(f"  Offset prior failed: {e}")
        preds["offset"] = None

    ws = sample[sample["well"] == wid]
    for _, row in ws.iterrows():
        ridx = int(row["row_idx"])
        if tvt_phys is not None:
            tvt_val = float(tvt_phys.iloc[ridx])
        else:
            blended = blend_hidden({m: p for m, p in preds.items() if p is not None})
            tvt_val = float(blended[ridx])
        rows.append({"id": row["id"], "tvt": tvt_val})

    print(f"  Added {len(ws)} rows for {wid}")


submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

print(f"\nDone: {len(submission)} rows saved to submission.csv")
print(submission.head(10))

## Submission sanity checks

In [ ]:
print("=" * 60)
print("Submission Statistics")
print("=" * 60)

print(submission["tvt"].describe())

print("\nMissing values:")
print(submission.isna().sum())

print("\nDuplicate IDs:")
print(submission["id"].duplicated().sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(
    submission["tvt"],
    bins=60,
    edgecolor="black",
    alpha=0.8
)
axes[0].set_title("Distribution of Submitted TVT Values")
axes[0].set_xlabel("TVT (ft)")
axes[0].set_ylabel("Count")

for wid in TEST_WELLS:
    sub_w = submission[submission["id"].str.startswith(wid)]
    axes[1].plot(np.arange(len(sub_w)), sub_w["tvt"].values, lw=1, label=wid)

axes[1].set_title("Predicted TVT Trajectory per Test Well")
axes[1].set_xlabel("Prediction Index")
axes[1].set_ylabel("TVT (ft)")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

print("\nSubmission shape:", submission.shape)
print("Expected rows:", len(sample))

assert len(submission) == len(sample), "Row count mismatch!"
assert submission["id"].nunique() == len(sample), "Duplicate IDs found!"
assert submission["tvt"].isna().sum() == 0, "NaNs found in submission!"

print("\n✓ Submission passed all checks")